In [10]:
from awattar_services import AwattarServices

# Create an instance of the class
awattar_services = AwattarServices()


ModuleNotFoundError: No module named 'awattar_services'

In [ ]:
import datetime
import timedelta

current_datetime = datetime.datetime.now() - timedelta(hours=24)
unix_timestamp = current_datetime.timestamp()
start_of_day, end_of_day = AwattarServices.get_start_and_end_of_day(unix_timestamp)
int(start_of_day.timestamp()*1000)
int(end_of_day.timestamp()*1000)

TypeError: 'module' object is not callable

In [ ]:
datetime.datetime.now()

datetime.datetime(2024, 10, 28, 1, 36, 13, 720798)

In [ ]:
import datetime
import timedelta

current_datetime = datetime.datetime.now() - timedelta(hours=24)
unix_timestamp = current_datetime.timestamp()
start_of_day, end_of_day = AwattarServices.get_start_and_end_of_day(unix_timestamp)
int(start_of_day.timestamp()*1000)
int(end_of_day.timestamp()*1000)

TypeError: 'module' object is not callable

In [ ]:
awattar_services.AWATTAR_ONE_DAY_PERIOD('1693526400000', '1696118399000')

,start_timestamp,end_timestamp,marketprice,unit
0,2023-09-01 00:00:00,2023-09-01 01:00:00,93.29,Eur/MWh
1,2023-09-01 01:00:00,2023-09-01 02:00:00,92.09,Eur/MWh
2,2023-09-01 02:00:00,2023-09-01 03:00:00,91.80,Eur/MWh
3,2023-09-01 03:00:00,2023-09-01 04:00:00,95.98,Eur/MWh
4,2023-09-01 04:00:00,2023-09-01 05:00:00,111.93,Eur/MWh
...,...,...,...,...
714,2023-09-30 18:00:00,2023-09-30 19:00:00,133.18,Eur/MWh
715,2023-09-30 19:00:00,2023-09-30 20:00:00,111.90,Eur/MWh
716,2023-09-30 20:00:00,2023-09-30 21:00:00,107.40,Eur/MWh
717,2023-09-30 21:00:00,2023-09-30 22:00:00,101.04,Eur/MWh


In [ ]:
import json
import datetime
import timedelta
# Get today and yesterday's date
today = datetime.datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
yesterday = today - timedelta(days=1)

# Convert to a UNIX timestamp (in seconds)
timestamp_start_yesterday = int(yesterday.timestamp())
timestamp_start_today = int(today.timestamp())

# If you need the timestamps in milliseconds, multiply by 1000
timestamp_start_yesterday_ms = timestamp_start_yesterday * 1000
timestamp_start_today_ms = timestamp_start_today * 1000

print(f"Timestamp for 00:00:00 yesterday: {timestamp_start_yesterday} seconds / {timestamp_start_yesterday_ms} milliseconds")
print(f"Timestamp for 24:00:00 yesterday (00:00:00 today): {timestamp_start_today} seconds / {timestamp_start_today_ms} milliseconds")


TypeError: 'module' object is not callable

In [ ]:
awattar_services.AWATTAR_ONE_DAY_PERIOD(timestamp_start_yesterday_ms, timestamp_start_today_ms)

TypeError: AWATTAR_ONE_DAY_PERIOD() missing 1 required positional argument: 'file_path'

In [ ]:
timestamp_start_yesterday_ms

1695942000000

In [37]:





import sys 
sys.path.append('/home/pi/smart_plug/')
import os
import pandas as pd
import requests
from datetime import datetime, timedelta
from main_services.common_utils import common_utils 
import pytz
import time 

class AwattarServices:
    def __init__(self):
        self.dataset_path = '/home/pi/smart_plug/dataset/awattar_data.csv'
        self.dataset_path_automode = '/home/pi/smart_plug/dataset/awattar_data_automode.csv'
        self.awattar_json_url = "https://api.awattar.at/v1/marketdata?start={}&end={}"
        # self.download_awattar_data()
        # if not os.path.isfile(self.dataset_path):
        #    raise Exception('Dataset not found')
        return None

    def check_market_price(self, eur: str):
        # This code runs whene current day max() price is less then 25% off
        creterion_timestamp = self.marketdata_df[
            self.marketdata_df['marketprice'] <= self.marketdata_df['marketprice'].max() * 0.75]
        # Get the current time
        current_time = datetime.datetime.now().time()
        print(current_time)
        print(creterion_timestamp)
        print(int(eur))
        # Iterate over the rows of the DataFrame
        for _, row in creterion_timestamp.iterrows():
            start_time = row['start_timestamp'].time()
            end_time = row['end_timestamp'].time()

            # Check if the current time is within the start and end time for the current row

            if start_time <= current_time <= end_time:
                print('Current time is between start and end time for row', _)
                self.bulb.bulb_on()
                break
                # bulb on
            else:
                print('Price is too high')
                self.bulb.bulb_off()
                continue

    def get_average_awattar_price_over_period(self, start_time, end_time):
        """
        REPORT 4:
        This function returns the average price for a given time period
        
        Inputs:
            start_time: str eg: '2023-07-25 15:00:00'
            end_time: str eg: '2023-07-25 17:00:00'
        
        Output: average_price: float
        
        Usage:
            awattar_service = AwattarService()
            awattar_service.report_4('2023-07-25 15:00:00', '2023-07-25 17:00:00') --> 0.123
            
        """
        # Define the format of the string
        try:
            start_time = datetime.datetime.strptime(start_time, '%Y-%m-%d %H:%M:%S')
        except:
            print('start_time is not in the correct format')
            raise ValueError('start_time is not in the correct format')

        try:
            end_time = datetime.datetime.strptime(end_time, '%Y-%m-%d %H:%M:%S')
        except:
            print('end_time is not in the correct format')
            raise ValueError('end_time is not in the correct format')

        # adding 1 hour to end_time to include the end_time

        end_time = end_time + datetime.timedelta(hours=1)
        df = pd.read_csv(self.dataset_path)
        df['start_timestamp'] = pd.to_datetime(df['start_timestamp'])
        df['end_timestamp'] = pd.to_datetime(df['end_timestamp'])
        filter_df = df[(df['start_timestamp'] >= start_time) & (df['end_timestamp'] <= end_time)]
        return filter_df['marketprice'].mean()

    def download_awattar_data(self):
        # New data
        url = "https://api.awattar.at/v1/marketdata"
        new_df = requests.get(url).json()
        new_df = pd.json_normalize(new_df['data'])
        new_df['start_timestamp'] = pd.to_datetime(new_df['start_timestamp'], unit='ms')
        new_df['end_timestamp'] = pd.to_datetime(new_df['end_timestamp'], unit='ms')

        # Check if the file exists
        if not os.path.isfile(self.dataset_path):
            new_df.to_csv(self.dataset_path, index=True)
            return None

        old_df = pd.read_csv(self.dataset_path)
        old_df['start_timestamp'] = pd.to_datetime(old_df['start_timestamp'])
        old_df['end_timestamp'] = pd.to_datetime(old_df['end_timestamp'])

        old_end_date = old_df['end_timestamp'].max()
        new_start_date = new_df['start_timestamp'].min()

        if old_end_date == new_start_date:
            print('No new data available')
            return None

        elif old_end_date < new_start_date:
            # Saving the new data
            print('New data available... saving the new data')
            resulting_df = pd.concat([old_df, new_df], ignore_index=True)
            resulting_df.to_csv(self.dataset_path, index=True)

    """ 
    Retrieves the past awattar price and generates the dataset
    Updated: 28.10.2024
    """
    # Get past data from awattar data
    def GET_AWATTAR_PAST_DATA(self):
        # Delete the awattar_data csv before creation
        Awattar_Data_Path = '/home/pi/smart_plug/dataset/'+common_utils.static_awattar_filename
        if os.path.exists(Awattar_Data_Path):
            os.remove(Awattar_Data_Path)

        # This return the datetime timestamp before 24hours
        # Replace this with your Unix timestamp
        timezone = 'Europe/Vienna'  # Replace with your desired timezone

        start_of_day, end_of_day = AwattarServices.pastStartAndEndDateForAwattar(timezone)
    
        # current_datetime = datetime.now() - timedelta(hours=24)
        # unix_timestamp = current_datetime.timestamp()
        # start_of_day, end_of_day = AwattarServices.get_start_and_end_of_day2(unix_timestamp)
 
        # Get Awattar Data
        json_url = self.awattar_json_url.format(start_of_day, end_of_day)

        print("Awattar Link", json_url)

        awattar_json_response = requests.get(json_url).json()
        awattar_json_response = pd.json_normalize(awattar_json_response['data'])
        awattar_json_response['start_timestamp'] = pd.to_datetime(awattar_json_response['start_timestamp'], unit='ms') + pd.Timedelta(hours=1)
        awattar_json_response['end_timestamp'] = pd.to_datetime(awattar_json_response['end_timestamp'], unit='ms') + pd.Timedelta(hours=1)

        if os.path.exists(self.dataset_path):
            os.remove(self.dataset_path)
        awattar_json_response.to_csv(self.dataset_path, index=False)
        print("*******************************")
        print("awattar_json_response", awattar_json_response)
        print("*******************************")
        return awattar_json_response


    def pastStartAndEndDateForAwattar(timezone='Europe/Vienna'):
        # Get current timestamp in seconds
        timestamp = time.time()

        # Convert timestamp to datetime object
        dt_object = datetime.utcfromtimestamp(timestamp)

        # Set the timezone to UTC
        dt_object_utc = pytz.utc.localize(dt_object)

        # Convert UTC to the specified local time zone
        dt_object_local = dt_object_utc.astimezone(pytz.timezone(timezone))

        # Calculate the start of the previous day
        start_of_day = (dt_object_local - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
        end_of_day = start_of_day.replace(hour=23, minute=59, second=59, microsecond=999999)

        # Convert datetime objects to Unix timestamps in milliseconds
        start_timestamp = int(start_of_day.timestamp()) * 1000
        end_timestamp = int(end_of_day.timestamp()) * 1000

        print(f"Start of the previous day: {start_of_day}")
        print(f"End of the previous day: {end_of_day}")

        return start_timestamp, end_timestamp


    """ 
    Retrieves the future awattar price and generates the AutoMode
    Updated: 28.10.2024
    """
    def AWATTAR_FUTURE_PRICE_AUTOMODE(self):

        timezone = 'Europe/Vienna'  # Replace with your desired timezone

        start_of_day, end_of_day = AwattarServices.get_start_and_end_of_day(timezone)
        print("*******************************")
        print(f"Awattar After Start of the day: {start_of_day}")
        print(f"Awattar After End of the day: {end_of_day}")
        print("*******************************")
        # Get Awattar Data
        json_url = self.awattar_json_url.format(start_of_day, end_of_day)
        print(json_url)
        
        awattar_json_response = requests.get(json_url).json()
        awattar_json_response = pd.json_normalize(awattar_json_response['data'])
        awattar_json_response['start_timestamp'] = pd.to_datetime(awattar_json_response['start_timestamp'], unit='ms') + pd.Timedelta(hours=1)
        awattar_json_response['end_timestamp'] = pd.to_datetime(awattar_json_response['end_timestamp'], unit='ms') + pd.Timedelta(hours=1)

        if os.path.exists(self.dataset_path_automode):
            os.remove(self.dataset_path_automode)
        awattar_json_response.to_csv(self.dataset_path_automode, index=False)
        print("*******************************")
        print("awattar_json_response", awattar_json_response)
        print("*******************************")
        return awattar_json_response
    
    def get_start_and_end_of_day(timezone='Europe/Vienna'):
        # Get current timestamp in seconds
        timestamp = time.time()

        # Convert timestamp to datetime object
        dt_object = datetime.utcfromtimestamp(timestamp)

        # Set the timezone to UTC
        dt_object_utc = pytz.utc.localize(dt_object)

        # Convert UTC to the specified local time zone
        dt_object_local = dt_object_utc.astimezone(pytz.timezone(timezone))

        # Get the start and end of the day
        start_of_day = dt_object_local.replace(hour=0, minute=0, second=0, microsecond=0)
        end_of_day = start_of_day.replace(hour=23, minute=59, second=59, microsecond=999999)

        # Get the start of the next day
        start_of_next_day = start_of_day + timedelta(days=1)

        # Convert datetime objects to Unix timestamps in milliseconds
        start_timestamp = int(start_of_day.timestamp()) * 1000
        end_timestamp = int(start_of_next_day.timestamp()) * 1000 - 1  # Subtract 1 millisecond

        print(f"Start of the day: {start_of_day}")
        print(f"End of the day: {end_of_day}")

        return start_timestamp, end_timestamp


if __name__ == "__main__":
    awattar_services = AwattarServices()
    awattar_services.GET_AWATTAR_PAST_DATA()
    timezone = 'Europe/Vienna'
    #awattar_services.pastStartAndEndDateForAwattar()
    #awattar_services.AWATTAR_FUTURE_PRICE_AUTOMODE()



Start of the previous day: 2024-10-27 00:00:00+01:00
End of the previous day: 2024-10-27 23:59:59.999999+01:00
Awattar Link https://api.awattar.at/v1/marketdata?start=1729983600000&end=1730069999000
*******************************
awattar_json_response        start_timestamp       end_timestamp  marketprice     unit
0  2024-10-27 00:00:00 2024-10-27 01:00:00        84.07  Eur/MWh
1  2024-10-27 01:00:00 2024-10-27 02:00:00        82.23  Eur/MWh
2  2024-10-27 02:00:00 2024-10-27 03:00:00        80.43  Eur/MWh
3  2024-10-27 03:00:00 2024-10-27 04:00:00        74.44  Eur/MWh
4  2024-10-27 04:00:00 2024-10-27 05:00:00        76.21  Eur/MWh
5  2024-10-27 05:00:00 2024-10-27 06:00:00        87.30  Eur/MWh
6  2024-10-27 06:00:00 2024-10-27 07:00:00        86.43  Eur/MWh
7  2024-10-27 07:00:00 2024-10-27 08:00:00        87.75  Eur/MWh
8  2024-10-27 08:00:00 2024-10-27 09:00:00        83.92  Eur/MWh
9  2024-10-27 09:00:00 2024-10-27 10:00:00        66.18  Eur/MWh
10 2024-10-27 10:00:00 2024-10-2